# How Good Is the Search? Vectors Alone vs Vectors + Metadata

Each ask is run twice over the posters: **vector search alone** (CLIP text vs poster embeddings) and **the same vector search restricted by what the
detectors found** (a label hop, a people count, ratings, an overlap test). Two kinds of check are printed:

* **Constraint check**, from the stored detector values: how many results have too few people, a rating of POSSIBLE or higher, an unhappy face. A filtered
  search should have none of these: that is a guarantee, not a surprise.
* **Independent check**, from the public TMDB 5000 dataset (not produced by Google's detectors): do the results really belong to the topic? Sports movies are
  found through TMDB's keywords, family and animated movies through its genres.
* **recall@K** compares the index's answer with an exact brute-force search over the same candidates.

Needs the CLIP embeddings (`python embed.py`) and internet once, for the TMDB metadata.

In [ ]:
import ast
import csv
import io
import os
import urllib.request
import zipfile

import pandas as pd

import config
import vector_search as vs

client = config.get_client()
K = 8
URL = "https://www.kaggle.com/api/v1/datasets/download/tmdb/tmdb-movie-metadata"
CACHE = "tmdb_5000_movies.csv"
if not os.path.exists(CACHE):
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL, timeout=120).read())) as z:
        open(CACHE, "wb").write(z.read("tmdb_5000_movies.csv"))
tmdb = {}
for r in csv.DictReader(open(CACHE, encoding="utf-8")):
    tmdb[r["id"]] = {"title": r["title"], "genres": [g["name"] for g in ast.literal_eval(r["genres"])],
                     "keywords": [k["name"].lower() for k in ast.literal_eval(r["keywords"])]}
SPORT_WORDS = ("sport", "football", "basketball", "baseball", "soccer", "boxing", "hockey", "tennis", "golf", "coach", "team",
               "athlete", "olympic", "wrestling", "racing", "swimming", "rugby", "player")


def stem(row):
    return row["gcs_object_name"].rsplit("/", 1)[-1].rsplit(".", 1)[0]


def is_sports(row):
    return any(w in k for k in tmdb.get(stem(row), {}).get("keywords", []) for w in SPORT_WORDS)


def is_family_or_animation(row):
    g = tmdb.get(stem(row), {}).get("genres", [])
    return "Family" in g or "Animation" in g


def evaluate(ask, variants, checks):
    rows_out, titles = [], {}
    for name, kw in variants.items():
        rows, _ = vs.search(client, ask, k=K, **kw)
        exact = vs.exact_search(client, ask, k=K, **kw)
        entry = {"ask": ask, "search": name, "results": len(rows)}
        for label, fn in checks.items():
            entry[label] = sum(bool(fn(r)) for r in rows)
        entry["recall@K"] = f"{vs.recall_at_k([r['image_id'] for r in rows], exact):.0%}"
        rows_out.append(entry)
        titles[(ask, name)] = ", ".join(tmdb.get(stem(r), {}).get("title", "?") for r in rows)
    return rows_out, titles

## Team Sports: a Label, a People Count and Ratings

In [ ]:
rated = lambda r: (r.get("adult_content_rank") or 1) > 2 or (r.get("violent_content_rank") or 1) > 2
checks = {"fewer than 3 people": lambda r: (r.get("face_count") or 0) < 3, "sports (TMDB keywords)": is_sports, "rated POSSIBLE+": rated}
variants = {"vector only": {},
            "vector + metadata": {"criteria": vs.people_criteria(3), "label": vs.SPORT_LABELS}}
table, titles = [], {}
for ask in ["a sports movie about a team", "a team celebrating together", "players on a field"]:
    t, ti = evaluate(ask, variants, checks)
    table += t
    titles.update(ti)
pd.DataFrame(table)

*fewer than 3 people* should be 0 for the filtered search by construction; *sports (TMDB keywords)* is the check that the filter did not drag the results off topic.

In [ ]:
for (ask, name), t in titles.items():
    print(f"{name:18} | {ask}\n    {t}\n")

## Overlap: a Player Holding a Ball

In [ ]:
touching = vs.overlapping_ids(client, first="person|man|woman|boy|girl", second="ball")
checks = {"no ball on a person": lambda r: r["image_id"] not in set(touching), "rated POSSIBLE+": rated}
variants = {"vector only": {}, "vector + overlap": {"criteria": vs.only_ids(touching)}}
t, _ = evaluate("a player holding a ball", variants, checks)
pd.DataFrame(t)

## Kid-Friendly Asks: Ratings and Faces

In [ ]:
unhappy = lambda r: (r.get("face_count") or 0) > 0 and (r.get("min_face_joy") or 0) < 0.6
checks = {"unhappy face": unhappy, "family or animation (TMDB genres)": is_family_or_animation, "rated POSSIBLE+": rated}
variants = {"vector only": {}, "vector + metadata": {"criteria": vs.family_criteria(2, 0.6)}}
table = []
for ask in ["something a child could watch tonight, where everyone looks happy", "a movie a child could watch", "a fun family comedy where everyone looks happy"]:
    t, _ = evaluate(ask, variants, checks)
    table += t
pd.DataFrame(table)